# Clase 9 · Material extra — Recetas de transfer learning

**Deep Learning para Cientistas Sociales**

En clase hicimos transfer learning **a mano**, para verle el motor. Acá está lo mismo
como se escribe en la vida real, con las bibliotecas que vas a encontrar en cualquier
tutorial o paper. Cada receta es **corta, completa y se puede copiar sola**.

| receta | qué hace | etiquetas que necesita | tarda (CPU) | baja |
|---|---|---|---|---|
| **1** | Nivel 1 en tres líneas con `sentence-transformers` | 50 a 500 | segundos | ~470 MB |
| **2** | Fine-tuning con el `Trainer` de Hugging Face (BETO) | 300 y más | ~1 min | ~440 MB |
| **3** | Zero-shot: clasificar describiendo la categoría | 0 para entrenar, **100+ para evaluar** | ~10 s | ~430 MB |
| **4** | Tu propio CSV: de un archivo tuyo a una tabla con puntajes | las que tengas | segundos | — |

Todas usan **la misma tarea, la misma partición y el mismo test** que el notebook de la
clase (*¿esta nota es sobre política?*), así que sus números se comparan directo con la
tabla de la sección 7. **Corré primero la sección 0**; después, cualquier receta en
cualquier orden.

> **Necesita red** (para bajar los modelos). En Colab anda tal cual. Los modelos se
> fijan con su **revisión exacta**: es lo que vas a citar en tu sección de métodos.

## 0. Preparar el entorno y los datos (corré esto siempre)

In [ ]:
# =============================================================================
# BOOTSTRAP — la misma celda en todos los notebooks del curso. No la edites.
# En Colab: clona el repo del curso (o usa el .zip que subas) e instala lo que falte.
# En tu máquina: encuentra el repo subiendo carpetas y no instala nada.
# =============================================================================
import os
import pathlib
import subprocess
import sys
import zipfile

REPO = "https://github.com/johannperez/dlss"
RAMA = "main"
CARPETA = "dlss"


def en_colab():
    """¿Estamos en Google Colab? Se lo preguntamos a Python, no lo adivinamos."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def buscar_raiz(desde=None):
    """Sube carpeta por carpeta hasta encontrar la que tiene `_shared/ssdl`."""
    punto = pathlib.Path(desde or os.getcwd()).resolve()
    for candidata in [punto, *punto.parents]:
        if (candidata / "_shared" / "ssdl").is_dir():
            return candidata
    return None


def raiz_desde_zip():
    """Plan B: si subiste el .zip del repo (dlss-main.zip o parecido), lo abre."""
    for archivo in sorted(pathlib.Path(".").glob("dlss*.zip")):
        with zipfile.ZipFile(archivo) as z:
            z.extractall(CARPETA + "-zip")
        for ssdl in pathlib.Path(CARPETA + "-zip").glob("**/_shared/ssdl"):
            return ssdl.parent.parent.resolve()
    return None


raiz = buscar_raiz()

if raiz is None and en_colab():
    if not pathlib.Path(CARPETA).exists():
        print("Bajando el repo del curso (una sola vez por sesión)...")
        clon = subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,
                               REPO, CARPETA],
                              env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
        if clon.returncode != 0:
            print("No pude clonar el repo. Pruebo con un .zip subido a mano...")
    raiz = buscar_raiz(CARPETA) if pathlib.Path(CARPETA).exists() else None
    raiz = raiz or raiz_desde_zip()

if raiz is None:
    raise SystemExit(
        "No encuentro el repo del curso.\n"
        "· En Colab, si el clone falló (por ejemplo, «Repository not found» o 404):\n"
        "  el repo github.com/johannperez/dlss tiene que ser público. Si no lo es,\n"
        "  o no hay red hacia GitHub, conseguí el .zip del repo (en la página del\n"
        "  repo: Code → Download ZIP; o pedíselo a quien da el curso), subilo con\n"
        "  el panel de archivos de Colab (el nombre tiene que empezar con `dlss`,\n"
        "  como dlss-main.zip) y volvé a correr esta celda.\n"
        "· En tu máquina: abrí este notebook desde adentro de la carpeta `dlss`.\n"
        "Más detalle en recursos/setup.md."
    )

# Dependencias. En Colab casi todas vienen puestas: instalamos sólo lo que falte.
# En tu máquina no instalamos nada (torch es pesado y varias clases no lo usan):
# si falta algo, avisamos y lo instalás vos (ver recursos/setup.md).
faltan = []
for modulo, paquete in [("numpy", "numpy"), ("matplotlib", "matplotlib"),
                        ("torch", "torch")]:
    try:
        __import__(modulo)
    except ImportError:
        faltan.append(paquete)
if faltan and en_colab():
    print(f"Instalando {', '.join(faltan)}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan],
                   check=True)
elif faltan:
    print(f"[!] Falta {', '.join(faltan)}. Si este notebook lo usa, instalalo con:\n"
          f"    {sys.executable} -m pip install {' '.join(faltan)}")

sys.path.insert(0, str(raiz / "_shared"))   # para poder hacer `import ssdl`
os.environ["DLSS_RAIZ"] = str(raiz)         # para que ssdl encuentre los datos

print(f"Listo. Repo del curso en: {raiz}")

In [ ]:
import subprocess
import sys

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from ssdl import datos, graficos as g, utils

# sentence-transformers no siempre viene instalada en Colab. Si falta, la instalamos.
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"],
                   check=True)

import torch
import transformers
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

utils.fijar_semilla(1983)

# La misma tarea, la misma partición y el mismo test que el notebook de la clase.
notas = datos.titulares_prensa()
titulares = [n["titular"] for n in notas]
y = np.array([int(n["tema"] == "Politica") for n in notas])
i_train, i_test = train_test_split(np.arange(len(y)), test_size=0.25,
                                   stratify=y, random_state=1983)


def muestra_de_entrenamiento(n, semilla=1983):
    # n titulares de train al azar, con la misma proporción de positivos.
    rng = np.random.default_rng(semilla)
    pos = [i for i in i_train if y[i] == 1]
    neg = [i for i in i_train if y[i] == 0]
    k = max(2, int(round(n * len(pos) / len(i_train))))
    return np.concatenate([rng.choice(pos, k, replace=False),
                           rng.choice(neg, n - k, replace=False)])


idx = muestra_de_entrenamiento(200)        # las mismas 200 etiquetas de la clase
textos_test = [titulares[i] for i in i_test]
print(f"Listo: {len(idx)} titulares para entrenar, {len(i_test)} de test.")

---

## Receta 1 — Nivel 1 en tres líneas: `sentence-transformers`

En la clase escribimos `vectorizar()` a mano: tokenizar, pasar por el modelo sin
gradiente, promediar las fichas, normalizar. La biblioteca `sentence-transformers`
hace exactamente eso en una línea, `modelo.encode(textos)`. Arriba va la misma
regresión logística de siempre.

**Cuándo usarla:** siempre primero. Es barata, estable, corre en CPU y la regresión
logística se le explica a cualquier revisor.

Más modelos de oraciones en castellano o multilingües: buscá *sentence-similarity* en
el Hub y filtrá por idioma (`es`). Cambiar de modelo es cambiar una línea.

In [ ]:
from sentence_transformers import SentenceTransformer

MODELO_EMB = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
REV_EMB = "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"

lector = SentenceTransformer(MODELO_EMB, revision=REV_EMB)                     # 1
X = lector.encode(titulares, batch_size=64, normalize_embeddings=True)         # 2
clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X[idx], y[idx])  # 3

pred = clf.predict(X[i_test])
print(f"Vectores: {X.shape}.  Receta 1, F1 en test: "
      f"{g.metricas_binarias(y[i_test], pred)['f1']:.2f}  (en clase, a mano: 0.64)")
g.frase_de_metricas(y[i_test], pred, fenomeno="nota sobre política");

---

## Receta 2 — Fine-tuning con el `Trainer` de Hugging Face

Lo que en clase fue un loop de cinco estaciones, acá lo esconde `Trainer`. Adentro
hace lo mismo, más contabilidad: lotes, *learning rate* que baja de a poco,
evaluación por época. Usamos **BETO** (BERT entrenado en castellano por la Universidad
de Chile), el caballito de batalla para texto en castellano.

Tres cosas que esta receta hace bien y que los tutoriales a veces no:

1. **Separa una validación** de las etiquetas de entrenamiento, para mirar cómo va
   época por época. El test **no se usa** hasta el final, una sola vez.
2. **Fija la semilla y la revisión del modelo.**
3. **No guarda checkpoints** (`save_strategy="no"`): en Colab se llena el disco rápido.

`AutoModelForSequenceClassification` = el modelo prestado + una capa nueva arriba con
`num_labels` salidas. Es el `Clasificador` que armamos en clase. Te va a avisar que la
capa `classifier` es nueva (*newly initialized*): es esperable, es la capa tuya.

> **Ojo con el tiempo.** BETO tiene 110 M de perillas y acá se mueven todas. Con
> `N_FT = 200` tardó 13 s en una laptop reciente con 2 hilos de CPU; en Colab sin GPU
> contá hasta un minuto. Con todas las etiquetas (1103) es cinco veces más: para eso,
> GPU (*Entorno de ejecución → Cambiar tipo de entorno → T4*).
>
> **Y mirá la F1 de validación época por época:** con 40 casos salta muchísimo de una
> época a otra. Es ruido de muestra chica, no señal. Por eso la evaluación seria se
> hace con 100-300 casos, no con 40.

In [ ]:
import tempfile

from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          Trainer, TrainingArguments)

MODELO_FT = "dccuchile/bert-base-spanish-wwm-uncased"        # BETO
REV_FT = "d1c9c4565c9d6731e57ed7f027b802697bad861e"
N_FT = 200          # con cuántas etiquetas (probá 50 y len(i_train))

tok_ft = AutoTokenizer.from_pretrained(MODELO_FT, revision=REV_FT)


class Titulares(torch.utils.data.Dataset):
    # Lo mínimo que Trainer necesita: tokens + etiqueta, uno por ejemplo.
    def __init__(self, indices):
        self.enc = tok_ft([titulares[i] for i in indices], truncation=True, max_length=64)
        self.y = [int(y[i]) for i in indices]

    def __len__(self):
        return len(self.y)

    def __getitem__(self, k):
        return {**{c: v[k] for c, v in self.enc.items()}, "labels": self.y[k]}


idx_ft = muestra_de_entrenamiento(N_FT)
idx_tr, idx_val = train_test_split(idx_ft, test_size=0.2, stratify=y[idx_ft],
                                   random_state=1983)      # validación: sale del TRAIN


def metricas(salida):
    pred = salida.predictions.argmax(-1)
    return {"f1": g.metricas_binarias(salida.label_ids, pred)["f1"]}


modelo_ft = AutoModelForSequenceClassification.from_pretrained(
    MODELO_FT, revision=REV_FT, num_labels=2)

args = TrainingArguments(
    output_dir=tempfile.mkdtemp(), num_train_epochs=3, learning_rate=5e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=64,
    eval_strategy="epoch", save_strategy="no", logging_strategy="epoch",
    report_to="none", seed=1983)

entrenador = Trainer(model=modelo_ft, args=args,
                     train_dataset=Titulares(idx_tr), eval_dataset=Titulares(idx_val),
                     data_collator=transformers.DataCollatorWithPadding(tok_ft),
                     compute_metrics=metricas)

with utils.medir_tiempo(f"fine-tuning de BETO con {len(idx_tr)} etiquetas"):
    entrenador.train()

# Recién ahora, una sola vez, el test.
pred_ft = entrenador.predict(Titulares(i_test)).predictions.argmax(-1)
print(f"\nReceta 2, F1 en test: {g.metricas_binarias(y[i_test], pred_ft)['f1']:.2f}")
g.frase_de_metricas(y[i_test], pred_ft, fenomeno="nota sobre política");

**Lo que medimos nosotros** (200 etiquetas, 160 para entrenar y 40 de validación):
F1 en test 0.56 a 0.58, según la máquina. Con el mismo código, en CPU y en GPU los
números cambian un poco: el orden de las cuentas no es idéntico.

**Para comparar en serio** (la lección de la clase): corré la receta 2 con
`seed=1984` y `seed=1985` en `TrainingArguments`. Si el F1 se mueve más que la
diferencia con la receta 1, no podés decir que una le gana a la otra. Reportá el
promedio y el rango, no la mejor corrida.

**Variantes que vas a ver en los tutoriales**, todas con este mismo esqueleto:
- otro modelo: cambiá `MODELO_FT` (por ejemplo, el MiniLM multilingüe de la receta 1,
  que es más chico y rápido);
- más de dos categorías: `num_labels=k` y etiquetas de 0 a k-1;
- congelar parte del modelo antes de entrenar, como en clase:
  `for p in modelo_ft.bert.parameters(): p.requires_grad = False`.

---

## Receta 3 — Zero-shot: cero etiquetas para entrenar

Un modelo entrenado en *inferencia*: le das un texto y una hipótesis ("Esta nota es
sobre política.") y dice si el texto la apoya. Con eso se clasifica **sin un solo
ejemplo de entrenamiento**: describís la categoría con palabras.

**Lo que no te ahorra: la evaluación.** Necesitás las mismas 100-300 etiquetas a mano
para saber si anda. Y una trampa: si probás cinco descripciones y te quedás con la que
mejor da **en el test**, ajustaste contra el test (clase 4). La descripción se elige
en una validación, o se fija antes de mirar.

In [ ]:
from transformers import pipeline

MODELO_ZS = "MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli"
REV_ZS = "0a71e92a985b6e1ad1828cf67ce9c459639c1dca"

zero_shot = pipeline("zero-shot-classification", model=MODELO_ZS, revision=REV_ZS)

CATEGORIAS = ["política", "otro tema"]            # fijadas ANTES de mirar el test
PLANTILLA = "Esta nota es sobre {}."

with utils.medir_tiempo(f"zero-shot sobre {len(textos_test)} titulares"):
    salidas = zero_shot(textos_test, candidate_labels=CATEGORIAS,
                        hypothesis_template=PLANTILLA, batch_size=32)
pred_zs = np.array([int(s["labels"][0] == "política") for s in salidas])

print(f"\nReceta 3, F1 en test: {g.metricas_binarias(y[i_test], pred_zs)['f1']:.2f}  "
      f"(cero etiquetas para entrenar)")
g.frase_de_metricas(y[i_test], pred_zs, fenomeno="nota sobre política");
print("\nAhora leé qué marcó como política sin serlo. ¿Qué entendió el modelo por «política»?")
falsos_positivos = [t for t, p, real in zip(textos_test, pred_zs, y[i_test])
                    if p == 1 and real == 0]
for t in falsos_positivos[:10]:
    print("   ·", t[:90])

**Leé esa lista con cuidado.** Varios de esos "errores" son política de verdad
—candidaturas, un alcalde que deja el cargo— y la etiqueta del corpus dice otro tema.
Cuando eso pasa, el F1 está midiendo tu desacuerdo con quien etiquetó, no sólo la
calidad del modelo. (Es el `TODO 6` del notebook completo, y la clase 10 entera.)

**Lo que medimos:** F1 0.39 con cero etiquetas, casi lo mismo que el diccionario de
palabras clave de la clase (0.40). Probá otra `PLANTILLA` u otras `CATEGORIAS`... sobre
**otra** muestra, no sobre este test.

---

## Receta 4 — Tu propio CSV

Lo que vas a hacer de verdad: un archivo con **todos** tus textos, de los cuales
etiquetaste algunos. La receta:

1. lee el CSV (una columna de texto y una de etiqueta, vacía donde no etiquetaste);
2. con las etiquetadas, separa entrenamiento y test y te da la tabla honesta: el piso
   y el nivel 1 con su frase;
3. entrena con todas las etiquetadas y le pone puntaje a **todas** las filas;
4. guarda un CSV nuevo con el puntaje al lado, para que ordenes y leas.

Acá la probamos con el corpus del curso, haciendo de cuenta que sólo etiquetamos 300.
Para usarla con lo tuyo, cambiá `RUTA`, `COLUMNA_TEXTO` y `COLUMNA_ETIQUETA`.

In [ ]:
import csv
import tempfile
import pathlib

# --- Armamos un CSV de ejemplo (en tu caso, este archivo ya lo tenés) ---
RUTA = pathlib.Path(tempfile.gettempdir()) / "mi-corpus.csv"
etiquetadas = set(muestra_de_entrenamiento(300, semilla=7).tolist())
with open(RUTA, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["texto", "es_politica"])
    for i, t in enumerate(titulares):
        w.writerow([t, y[i] if i in etiquetadas else ""])

COLUMNA_TEXTO = "texto"
COLUMNA_ETIQUETA = "es_politica"      # 1 / 0, vacía si no la etiquetaste

# --- 1. Leer ---
with open(RUTA, encoding="utf-8") as f:
    filas = list(csv.DictReader(f))
textos = [r[COLUMNA_TEXTO] for r in filas]
con_etiqueta = [k for k, r in enumerate(filas) if r[COLUMNA_ETIQUETA].strip() != ""]
etiqueta = np.array([int(filas[k][COLUMNA_ETIQUETA]) for k in con_etiqueta])
print(f"{len(textos)} textos, {len(con_etiqueta)} etiquetados "
      f"({etiqueta.mean():.0%} positivos).")

# --- 2. Evaluar: entrenamiento y test SÓLO entre las etiquetadas ---
V = lector.encode(textos, batch_size=64, normalize_embeddings=True)   # el de la receta 1
Ve = V[con_etiqueta]
k_tr, k_te = train_test_split(np.arange(len(con_etiqueta)), test_size=0.3,
                              stratify=etiqueta, random_state=1983)
clf_csv = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Ve[k_tr], etiqueta[k_tr])
pred_csv = clf_csv.predict(Ve[k_te])
print(f"\nPiso (decir siempre la clase mayoritaria): accuracy "
      f"{max(etiqueta[k_te].mean(), 1 - etiqueta[k_te].mean()):.2f}, F1 0.00")
g.frase_de_metricas(etiqueta[k_te], pred_csv, fenomeno="positivo");
print(f"(Con sólo {len(k_te)} casos de test, este número tiene un margen de error grande.)")

# --- 3. Entrenar con todas las etiquetadas y puntuar todo ---
clf_final = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Ve, etiqueta)
puntaje = clf_final.predict_proba(V)[:, 1]

# --- 4. Guardar ---
SALIDA = RUTA.with_name("mi-corpus-con-puntaje.csv")
with open(SALIDA, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow([COLUMNA_TEXTO, COLUMNA_ETIQUETA, "puntaje"])
    for r, p in sorted(zip(filas, puntaje), key=lambda z: -z[1]):
        w.writerow([r[COLUMNA_TEXTO], r[COLUMNA_ETIQUETA], f"{p:.3f}"])
print(f"\nGuardado: {SALIDA}  (ordenado de mayor a menor puntaje: leé los primeros)")

---

## Para ver más implementaciones

Ordenadas de la más cercana a esta clase a la más lejana. Todas verificadas el
2026-11-13.

| qué | dónde | para qué |
|---|---|---|
| **Curso de Hugging Face, capítulo 3** (fine-tuning con `Trainer` y a mano), **en castellano** | https://huggingface.co/learn/llm-course/es/chapter3/1 | la receta 2 explicada paso a paso; el capítulo 2 explica tokenizadores y modelos |
| Guía oficial de clasificación de texto con `transformers` | https://huggingface.co/docs/transformers/tasks/sequence_classification | la receta 2 completa, con `datasets` y `evaluate` |
| Documentación de `sentence-transformers` | https://sbert.net/docs/sentence_transformer/usage/usage.html | la receta 1: buscar por significado, agrupar, comparar textos |
| **SetFit**: fine-tuning con muy pocas etiquetas (8 a 64 por clase) | https://github.com/huggingface/setfit | el punto medio entre la receta 1 y la 2, pensado para cuando etiquetar es caro |
| Zero-shot en el Hub | https://huggingface.co/tasks/zero-shot-classification | la receta 3, con demos y otros modelos |
| BETO | https://huggingface.co/dccuchile/bert-base-spanish-wwm-uncased | la ficha del modelo: con qué texto se entrenó (eso es lo que tomás prestado) |
| `pysentimiento` | https://github.com/pysentimiento/pysentimiento | modelos ya afinados en castellano: sentimiento, emociones, discurso de odio, ironía |
| Transfer learning con imágenes, en PyTorch | https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html | la misma idea con una CNN (clase 5): congelar y cambiar la última capa |
| Métricas de evaluación en `scikit-learn` | https://scikit-learn.org/stable/modules/model_evaluation.html | para escribir tu propio `reportar()` |

Y para el **criterio** (qué reportar, cómo validar), `referencias.md` en la carpeta de
la clase: empezá por Barberá et al. (2021).